In [ ]:
import os
import sys
import time
import yaml
import pandas as pd

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt

In [8]:
MODEL = "text-embedding-3-small"
input_cost = 0.01 # batch cost per 1M tokens
bytes_per_request = 4 * EMBEDDING_DIMENSION

In [9]:
START_IDX = 0
END_IDX = 800
REPLACE = False
MODE = 'WRITE' # ESTIMATE | BATCH | WRITE

In [10]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data.parquet"))

In [11]:
rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
embedding_db_path = os.path.join(rel_path, "embeddings.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
embedding_db_conn = llt.get_embedding_store_db_conn(embedding_db_path)

input_tokens = 0
max_input_token_length = 0
max_input_token_id = ('','')
n_requests = 0
n_direct = 0
prompts_to_submit = []
embeddings_df = []

t0 = time.time()
for idx, row in df.iterrows():
    if idx < START_IDX or idx > END_IDX:
        continue
    date = row["date"]
    year = row["year"]
    item_no = row["item_no"]
    out_row = {"date": date, "year": year, "item_no": item_no}

    prompt = row["agenda_summary"]

    # check if prompt is in cache
    prompt_hash = llt.get_hash(prompt)
    cached_response = llt.check_embedding_cache(prompt_hash, embedding_db_conn)
    if (not REPLACE) and cached_response:
        if MODE=='WRITE':
            embedding = llt.get_embedding(prompt, MODEL, embedding_db_conn, overwrite=REPLACE)
            out_row["embedding"] = embedding
            embeddings_df.append(out_row)
        continue

    prompt_tokens = llt.count_tokens(prompt, MODEL)
    input_tokens += prompt_tokens
    if prompt_tokens > max_input_token_length:
        max_input_token_length = prompt_tokens
        max_input_token_id = (date, item_no)
        max_input_token_prompt = prompt
    n_requests += 1

    # add prompt to batch and check if batch is ready to submit
    if MODE=='BATCH':
        prompts_to_submit.append(prompt)
    elif MODE=='WRITE':
        embedding = llt.get_embedding(prompt, MODEL, embedding_db_conn, overwrite=REPLACE)
        out_row["embedding"] = embedding
        embeddings_df.append(out_row)
        n_direct += 1    

# submit prompts to batch
if MODE=='BATCH':
    if prompts_to_submit:
        input_filename = f"embeddings_batch.jsonl"
        llt.create_embedding_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, embedding_db_conn=embedding_db_conn, overwrite=REPLACE)
        prompts_to_submit = []
        print(f"    batch submitted")
elif MODE=='WRITE':
    out_df = pd.DataFrame(embeddings_df)
    out_df.to_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "embeddings.parquet"))

t1 = time.time()
print(f"    elapsed time: {(t1-t0)/60:.2f} minutes")

batch_db_conn.close()
embedding_db_conn.close()

print(f"{n_direct:,} direct requests made")

    elapsed time: 0.05 minutes
3 direct requests made


In [12]:
# Cost Estimate

total_input_cost = input_tokens / 1e6 * input_cost
total_output_size = n_requests * bytes_per_request

print(f"Estimated number of requests: {n_requests:,}")
print(f"Estimated total input tokens: {input_tokens:,}")
print(f"Estimated total input cost: ${total_input_cost:.2f}")
print(f"Estimated total output size: {total_output_size / 1e9:.2f} gb")
print(f"Maximum input token length: {max_input_token_length} (at: {max_input_token_id})")

Estimated number of requests: 3
Estimated total input tokens: 310
Estimated total input cost: $0.00
Estimated total output size: 0.00 gb
Maximum input token length: 136 (at: ('2022-02-10', '7'))
